# NumPy random walks

A reference for simulating and plotting multiple random walks.

## I need to

Simulate several walks, inspect their positions over time, and plot each walk as a separate line.

## What data do I have?

- `random_walk`: one Python list of positions.
- `all_walks`: a list containing many walks.
- `np_aw`: a 2-D NumPy array shaped `(walks, steps)`.

Matplotlib reads each **column** of a 2-D array as a line. Since `np_aw` has walks in rows, transpose it so each walk becomes a column.

## Determine the tool

Use `np.random.randint` to roll random integers, `np.array` to create a numeric 2-D array, `.shape` to inspect its dimensions, `.T` (or `np.transpose`) to swap axes, and `matplotlib.pyplot` to plot.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(123)  # makes this example reproducible

all_walks = []
for _ in range(5):
    random_walk = [0]
    for _ in range(100):
        step = random_walk[-1]
        dice = np.random.randint(1, 7)
        if dice <= 2:
            step = max(0, step - 1)
        elif dice <= 5:
            step += 1
        else:
            step += np.random.randint(1, 7)
        random_walk.append(step)
    all_walks.append(random_walk)

np_aw = np.array(all_walks)
print("shape before transpose:", np_aw.shape)

np_aw_t = np_aw.T
print("shape after transpose:", np_aw_t.shape)

plt.plot(np_aw_t)
plt.xlabel("Throw number")
plt.ylabel("Position")
plt.title("Five random walks")
plt.show()

## I need to

**Estimate the chance that a random walk reaches 60 steps high.**

Run the complete random walk many times, collect each final position, and calculate the proportion that finish at 60 or higher. This is a simulation-based probability estimate.

## What data do I have?

- One walk: a list of positions, including its start at `0`.
- Many walks: `all_walks`, a list of those lists.
- Final positions: the last value from every walk.

Each completed walk counts as one trial. A trial is a success when its final position is `>= 60`.

## Determine the tool

Use an outer loop (or list comprehension) to repeat the full walk, `random_walk[-1]` to collect its final position, and `np.mean(final_positions >= 60)` to calculate the fraction of successful trials. Use `plt.hist(...)` to see the distribution of final positions.

In [ ]:
def simulate_walk(number_of_throws=100):
    random_walk = [0]
    for _ in range(number_of_throws):
        step = random_walk[-1]
        dice = np.random.randint(1, 7)
        if dice <= 2:
            step = max(0, step - 1)
        elif dice <= 5:
            step += 1
        else:
            step += np.random.randint(1, 7)
        random_walk.append(step)
    return random_walk

number_of_simulations = 10_000
final_positions = np.array([
    simulate_walk()[-1] for _ in range(number_of_simulations)
])

chance_finish_at_least_60 = np.mean(final_positions >= 60)
print(f"Chance of finishing at 60 or higher: {chance_finish_at_least_60:.1%}")

plt.hist(final_positions, bins=20)
plt.axvline(60, color="red", linestyle="--", label="60 steps")
plt.xlabel("Final position after 100 throws")
plt.ylabel("Number of simulations")
plt.title("Distribution of final positions")
plt.legend()
plt.show()

## Important wording: finish at 60 vs. ever reach 60

The calculation above answers: **What is the chance the walk finishes at 60 or higher?** If the lab instead means **ever reaches 60 at any point**, keep every walk and test whether any position meets the target:

```python
walks = [simulate_walk() for _ in range(number_of_simulations)]
chance_ever_reach_60 = np.mean([max(walk) >= 60 for walk in walks])
```

Read the wording carefully: final-position probability and ever-reached probability are different questions.

## Quick checks

- `np_aw.shape` should be `(5, 101)`: five walks and the starting position plus 100 throws.
- `np_aw_t.shape` should be `(101, 5)`.
- More simulations usually make the estimated chance more stable.
- Remove the seed when you want a new random result on each run.